# RASopathy Functional Evidence Atlas

**How much functional evidence underpins ClinGen RASopathy VCEP variant classifications, and what is missing when a classification cannot be reached.**

Fernando S. R. — reproducible from public ClinVar and MaveDB data. Runs on Google Colab; no local setup required.

---

## Summary of findings

Across the 14 genes covered by the ClinGen RASopathy Variant Curation Expert Panel:

1. **Reclassification is close to static.** Between the January 2022 and August 2026 ClinVar releases, 20 of 1,483 missense VUS reached a confident classification while 26 moved the other way. The largest single destination was conflicting interpretation (175 variants). The missense cohort grew from 2,378 to 8,457 over the same period.

2. **Functional evidence is one-directional.** In the panel's own ClinVar submissions, functional criteria appear in roughly 55% of pathogenic classifications and 3% of benign ones — two BS3 applications in total.

3. **Curation coverage tracks familiarity, not burden.** LZTR1 contributes 2,150 missense entries with 12 curated; CBL contributes 1,316 with none. Together they are 41% of the cohort.

4. **Forty variants are curated and still uncertain.** Most carry no applied functional evidence.

5. **A coordinate convention hid deposited MAVE data.** An HGVS-string lookup against MaveDB returned zero coverage of the blocked set. That was wrong: the HRAS score sets number positions against the mature protein. After correcting the offset, two blocked VUS are present with quantitative scores. RAF1 and BRAF carry larger offsets because those depositions cover isolated domains.

6. **The recovered data is not usable as-is.** The source assay is a bacterial two-hybrid system with no membrane and a single effector, validated by ITC with explicitly greater variance near wild-type binding — which is where both recovered variants sit.

## Errors found in this pipeline that changed conclusions

Kept visible on purpose. Each was caught by instrumentation added after the fact, and a reader should know which results were once reported differently.

| Error | Effect | Fix |
|---|---|---|
| Missense regex admitted `p.Arg498Ter` | Nonsense counted as missense, inflating the reclassification count the study design rested on | Both residues constrained to the 20 standard amino acids |
| Criteria matched on a ±90-character window | 46% of ACMG code mentions unresolvable | Each code scoped to its own clause; unresolved share fell to 6% |
| Top-N truncation when reporting criteria frequency | BS3 read as absent when it was 2 | Report the crosstab, never the head of a sorted list |
| Blocked set defined by extraction success | Two variants dropped for being hard to parse | Membership from curation status; extraction annotates, does not select |
| MaveDB matched across genes | A string from an HRAS scan attributed to LZTR1 | Gene-scoped target sets |
| MaveDB lookup by HGVS string | Zero coverage reported where two variants exist | Coordinate offset resolved before matching |

## Measurement bias, stated up front

Detection of functional evidence relies on vocabulary in title, abstract and submission free text. It undercounts, and the undercount runs in the direction of this project's thesis. Where a claim benefits from its own measurement error, that is noted in the limitations.

In [ ]:
# ---------------------------------------------------------------------
# Configuration -- Google Colab
# Runtime ~40-60 min, dominated by ClinVar downloads (~900 MB).
# Set MOUNT_DRIVE = True to cache the downloads across sessions; otherwise
# they are re-fetched every time the runtime is recycled.
# ---------------------------------------------------------------------
from __future__ import annotations
import gzip, io, json, math, re, sys, time, urllib.parse, urllib.request
from pathlib import Path
import pandas as pd
import requests

MOUNT_DRIVE = False

PLATFORM = "colab"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = Path("/content/drive/MyDrive/rasopathy_atlas")
else:
    WORKDIR = Path("/content/rasopathy_atlas")
WORKDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIRS = [WORKDIR]
print(f"[env  ] {PLATFORM}; working directory {WORKDIR}")
if not MOUNT_DRIVE:
    print("[note ] downloads are not persisted; set MOUNT_DRIVE = True to cache them")

## 1. Constants and helpers

`MISSENSE_RE` constrains both residues to the 20 standard amino acids, so `p.Arg498Ter` and `p.Ile56Ile` cannot match. Truncating variants are resolved by PVS1 and were never candidates for functional-evidence rescue; admitting them inflated the reclassification count.

In [ ]:
CLINVAR_FTP = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/"
ARCHIVE_DIR = CLINVAR_FTP + "archive/"
MAVEDB      = "https://api.mavedb.org/api/v1"
UNIPROT     = "https://rest.uniprot.org/uniprotkb"
UA = {"User-Agent": "RASopathyAtlas/1.0 (research; public data only)"}
SLEEP = 0.3

OLD_YEAR, OLD_MONTH = 2022, 1
NEW_YEAR, NEW_MONTH = 2026, 8
ASSEMBLY = "GRCh38"

GENES = ["PTPN11", "SOS1", "RAF1", "RIT1", "KRAS", "NRAS", "HRAS",
         "BRAF", "MAP2K1", "MAP2K2", "SHOC2", "CBL", "LZTR1", "RRAS2"]
UNIPROT_BY_GENE = {"HRAS": "P01112", "RAF1": "P04049", "BRAF": "P15056"}

SCORE_DIR = WORKDIR / "mavedb_scores"; SCORE_DIR.mkdir(exist_ok=True)
OUT_DIR   = WORKDIR / "outputs";       OUT_DIR.mkdir(exist_ok=True)

AA3 = ["Ala","Arg","Asn","Asp","Cys","Gln","Glu","Gly","His","Ile",
       "Leu","Lys","Met","Phe","Pro","Ser","Thr","Trp","Tyr","Val"]
AA3_TO_1 = dict(zip(AA3, "ARNDCQEGHILKMFPSTWYV"))
AA1_TO_3 = {v: k for k, v in AA3_TO_1.items()}
_AA = "|".join(AA3)
MISSENSE_RE = re.compile(rf"p\.({_AA})(\d+)({_AA})(?![a-zA-Z])")
P_RE = re.compile(rf"p\.({_AA})(\d+)({_AA})")
C_RE = re.compile(r"c\.[\d_]+[ACGT]>[ACGT]")

USECOLS = ["Type","Name","GeneSymbol","ClinicalSignificance","LastEvaluated",
           "Assembly","ReviewStatus","NumberSubmitters","VariationID"]

FUNCTIONAL = {"PS3", "BS3"}
APPLIED_LOOSE = {"APPLIED", "APPLIED_BARE"}

# Frozen results act as tripwires. A mismatch is not necessarily an error --
# ClinVar changes -- but it must be seen before anything is written up.
EXPECTED = {
    "cohort_2022_01": 2378,
    "cohort_2026_08": 8457,
    "vcep_in_cohort": 215,
    "vcep_parseable": 213,
    "blocked_curated": 40,
    "blocked_with_codes": 38,
    "blocked_with_applied": 36,
    "plp_functional": 63, "plp_curated": 115,
    "blb_functional": 2,  "blb_curated": 60,
    "vus_functional": 5,  "vus_curated": 36,
}
DRIFT = []

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def cached(filename):
    for base in CACHE_DIRS:
        if not base.exists():
            continue
        hits = sorted(base.rglob(filename))
        if hits:
            return hits[0]
    return None


def fetch(url, filename):
    hit = cached(filename)
    if hit is not None and hit.stat().st_size > 0:
        print(f"[cache] {filename} ({hit.stat().st_size/1e6:.1f} MB)")
        return hit
    dest = WORKDIR / filename
    print(f"[get  ] {url}")
    urllib.request.urlretrieve(url, dest)
    print(f"[ok   ] {filename} ({dest.stat().st_size/1e6:.1f} MB)")
    return dest


def get_json(url, params=None, timeout=60):
    try:
        r = requests.get(url, params=params, headers=UA, timeout=timeout)
        ct = r.headers.get("Content-Type", "")
        return r.status_code, (r.json() if ct.startswith("application/json") else None), r
    except Exception as exc:
        print(f"[fail ] {type(exc).__name__}: {exc}")
        return None, None, None


def list_releases(base):
    with urllib.request.urlopen(base) as resp:
        html = resp.read().decode("utf-8", errors="ignore")
    return sorted(set(re.findall(r'href="(variant_summary_[^"]+\.txt\.gz)"', html)))


def resolve_release(year, month):
    # archive/YYYY/ is tried first: the archive root also holds recent
    # releases, so checking it first makes the year directory unreachable.
    target = f"variant_summary_{year}-{month:02d}.txt.gz"
    for base in (f"{ARCHIVE_DIR}{year}/", ARCHIVE_DIR):
        try:
            if target in list_releases(base):
                print(f"[pick ] {target}")
                return base + target, target
        except Exception as exc:
            print(f"[warn ] cannot list {base}: {exc}")
    raise RuntimeError(f"{target} not found in the ClinVar archive")


def bucket(sig):
    if not isinstance(sig, str):
        return "other"
    s = sig.lower()
    if "conflicting" in s: return "conflicting"
    if "uncertain"   in s: return "VUS"
    if "benign"      in s: return "B/LB"      # before pathogenic: "likely benign"
    if "pathogenic"  in s: return "P/LP"
    return "other"


def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0, 0.0)
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (p, max(0.0, c-h), min(1.0, c+h))


def check(name, actual, key):
    exp = EXPECTED[key]
    if actual != exp:
        DRIFT.append(f"{name}: got {actual:,}, expected {exp:,}")
        print(f"[DRIFT] {name}: got {actual:,}, expected {exp:,}")
    else:
        print(f"[ok   ] {name} = {actual:,}")

print("[init ] constants and helpers loaded")

## 2. ClinVar snapshots

Both sides of the comparison are frozen monthly archive releases. The undated weekly file would make the result irreproducible.

In [ ]:
def load_snapshot(path, genes=None):
    genes = genes or GENES
    keep = []
    for chunk in pd.read_csv(path, sep="\t", usecols=USECOLS, dtype=str,
                             chunksize=500_000, low_memory=False, on_bad_lines="skip"):
        hit = chunk[chunk["GeneSymbol"].isin(genes) & (chunk["Assembly"] == ASSEMBLY)]
        if len(hit):
            keep.append(hit)
    df = pd.concat(keep, ignore_index=True)
    df = df[df["Type"].str.contains("single nucleotide", case=False, na=False)].copy()
    parsed = df["Name"].fillna("").str.extract(MISSENSE_RE)
    parsed.columns = ["aa_ref", "aa_pos", "aa_alt"]
    df = pd.concat([df, parsed], axis=1)
    df = df[df["aa_ref"].notna() & (df["aa_ref"] != df["aa_alt"])]
    n_dup = int(df["VariationID"].duplicated().sum())
    if n_dup:
        print(f"[warn ] {n_dup} duplicate VariationIDs dropped")
    df = df.drop_duplicates("VariationID").copy()
    df["bucket"] = df["ClinicalSignificance"].map(bucket)
    df["hgvs_p_short"] = (df["aa_ref"].map(AA3_TO_1) + df["aa_pos"]
                          + df["aa_alt"].map(AA3_TO_1))
    print(f"[load ] {Path(path).name}: {len(df):,} true missense")
    return df

old_url, old_name = resolve_release(OLD_YEAR, OLD_MONTH)
new_url, new_name = resolve_release(NEW_YEAR, NEW_MONTH)
old = load_snapshot(fetch(old_url, old_name))
new = load_snapshot(fetch(new_url, new_name))

check("cohort_2022_01", len(old), "cohort_2022_01")
check("cohort_2026_08", len(new), "cohort_2026_08")

## 3. Is there anything to predict?

The original design was a temporal validation: rank VUS computationally, test against later reclassification. This measures the base rate first, before any effort is spent.

In [ ]:
j = old[["VariationID","GeneSymbol","bucket","Name"]].merge(
    new[["VariationID","bucket","ReviewStatus","LastEvaluated"]],
    on="VariationID", suffixes=("_old","_now"), validate="one_to_one")

print("== transition matrix (rows 2022-01, columns 2026-08) ==")
print(pd.crosstab(j["bucket_old"], j["bucket_now"], margins=True))

vus_base    = int((j["bucket_old"] == "VUS").sum())
resolved    = int(((j["bucket_old"] == "VUS") & j["bucket_now"].isin(["P/LP","B/LB"])).sum())
regressed   = int((j["bucket_old"].isin(["P/LP","B/LB"]) & (j["bucket_now"] == "VUS")).sum())
to_conflict = int(((j["bucket_old"] == "VUS") & (j["bucket_now"] == "conflicting")).sum())
new_entries = len(new) - len(j)

print(f"\nVUS at baseline          : {vus_base:,}")
print(f"VUS -> resolved          : {resolved}  ({resolved/vus_base:.1%})")
print(f"resolved -> VUS          : {regressed}   <- net flow is negative")
print(f"VUS -> conflicting       : {to_conflict}")
print(f"entries new since 2022   : {new_entries:,} ({new_entries/len(new):.0%} of today's cohort)")

print("\nper gene, current release:")
share = pd.crosstab(new["GeneSymbol"], new["bucket"], normalize="index")
gene_now = pd.concat([new["GeneSymbol"].value_counts().rename("n_missense"),
                      (share.get("VUS", 0)*100).round(1).rename("pct_VUS")], axis=1)
print(gene_now.sort_values("n_missense", ascending=False))
gene_now.to_csv(OUT_DIR / "gene_cohort_summary.csv")

Roughly 1.3% of VUS resolved over four and a half years, more classifications regressed into uncertainty than left it, and the dominant destination was conflicting interpretation. There is no reclassification flow to model. What follows is descriptive.

### Provenance of today's confident labels

The larger alternative cohort, and the reason it is not used for the headline: submissions made after 2022 very likely applied PP3 using REVEL or AlphaMissense, so computational features leak directly into those labels.

In [ ]:
old_ids  = set(old["VariationID"])
old_conf = set(old.loc[old["bucket"].isin(["P/LP","B/LB"]), "VariationID"])

conf = new[new["bucket"].isin(["P/LP","B/LB"])].copy()
conf["origin"] = conf["VariationID"].map(
    lambda v: "labelled_by_2022" if v in old_conf
    else ("in_2022_unlabelled" if v in old_ids else "absent_in_2022"))
print(pd.crosstab(conf["origin"], conf["bucket"], margins=True))
holdout = conf[conf["origin"] != "labelled_by_2022"]
print(f"\ncandidate held-out set: {len(holdout):,} (not used -- see note above)")

## 4. How much expert curation exists?

`variant_summary` carries no submitter column, so this requires `submission_summary`. A three-star review status indicates *some* recognised expert panel, not necessarily the RASopathy VCEP, so attribution is matched on the submitter string and the matched strings are printed for inspection.

In [ ]:
VCEP_PATTERN = re.compile(r"rasopathy", re.IGNORECASE)

def load_vcep_submissions():
    path = fetch(CLINVAR_FTP + "submission_summary.txt.gz", "submission_summary.txt.gz")
    header, skip = None, 0
    with gzip.open(path, "rt", errors="ignore") as fh:
        for i, line in enumerate(fh):
            if line.startswith("#"):
                header, skip = line.lstrip("#").rstrip("\n").split("\t"), i + 1
            else:
                break
    keep = []
    for chunk in pd.read_csv(path, sep="\t", dtype=str, names=header, skiprows=skip,
                             chunksize=500_000, low_memory=False, on_bad_lines="skip"):
        hit = chunk[chunk["Submitter"].fillna("").str.contains(VCEP_PATTERN)]
        if len(hit):
            keep.append(hit)
    df = pd.concat(keep, ignore_index=True)
    print(f"[load ] VCEP submissions: {len(df):,}")
    return df

subs = load_vcep_submissions()
print("\nsubmitter strings matched (verify by eye):")
print(subs["Submitter"].value_counts())

in_cohort = subs[subs["VariationID"].isin(set(new["VariationID"]))]
check("vcep_in_cohort", in_cohort["VariationID"].nunique(), "vcep_in_cohort")

vg = in_cohort.merge(new[["VariationID","GeneSymbol","bucket"]],
                     on="VariationID", how="left").drop_duplicates("VariationID")
print("\nby gene:");  print(vg["GeneSymbol"].value_counts())
print("\nby class:"); print(vg["bucket"].value_counts())
print("\ncoverage: {:.1%} of the missense cohort".format(
    in_cohort["VariationID"].nunique() / len(new)))

## 5. Which ACMG criteria were applied?

Criteria codes appear in submission free text, but records enumerate what was *evaluated*, marking each met or not met. A naive regex therefore counted BA1 on variants that remain VUS — impossible, since BA1 is stand-alone benign. Scoping each code to its own clause fixed it: unresolved mentions fell from 46% to 6%, and BA1-applied-on-VUS to zero.

`APPLIED_BARE` marks clauses listing codes with no verb. The convention suggests these are the applied set, but that is an assumption, so strict and inclusive counts are reported separately.

In [ ]:
CODE_RE = re.compile(r"\b(P(?:VS|S|M|P)\d|B(?:A|S|P)\d)"
                     r"(_(?:Very[ _]?Strong|Strong|Moderate|Supporting))?\b", re.IGNORECASE)
CLAUSE_SPLIT_RE = re.compile(r"(?<=[.;|])\s+|\n+|\s+[-\u2022]\s+")
NEGATION_RE = re.compile(r"\b(not\s+met|not\s+applied|not\s+applicable|not\s+invoked|"
                         r"not\s+used|does\s+not|did\s+not|was\s+not|were\s+not|cannot|"
                         r"could\s+not|insufficient|no\s+evidence|unable|fails?\s+to|"
                         r"excluded?|n/?a)\b", re.IGNORECASE)
AFFIRM_RE = re.compile(r"\b(met|applied|applies|invoked|satisfied|meets|therefore|thus)\b",
                       re.IGNORECASE)

def clause_mentions(text):
    out = []
    if not isinstance(text, str) or not text.strip():
        return out
    for clause in CLAUSE_SPLIT_RE.split(text):
        clause = clause.strip()
        if not clause:
            continue
        codes = list(CODE_RE.finditer(clause))
        if not codes:
            continue
        neg, aff = NEGATION_RE.search(clause), AFFIRM_RE.search(clause)
        if   neg and not aff:     verdict = "NOT_MET"
        elif aff and not neg:     verdict = "APPLIED"
        elif not neg and not aff: verdict = "APPLIED_BARE"
        else:                     verdict = "UNCLEAR"
        for m in codes:
            base = m.group(1).upper()
            mod = (m.group(2) or "").replace(" ", "_").title().replace("Very_Strong","VeryStrong")
            out.append({"code": base + mod, "base": base, "verdict": verdict,
                        "n_codes_in_clause": len(codes), "clause": clause[:300]})
    return out

text_cols = [c for c in ("Description","ExplanationOfInterpretation") if c in subs.columns]
subs["_text"] = subs[text_cols].fillna("").agg(" | ".join, axis=1)

rows = []
for _, r in subs.iterrows():
    for m in clause_mentions(r["_text"]):
        rows.append({"VariationID": r["VariationID"], "SCV": r.get("SCV"), **m})
men = pd.DataFrame(rows).merge(new[["VariationID","GeneSymbol","bucket"]],
                               on="VariationID", how="left")
men["in_cohort"] = men["bucket"].notna()
coh = men[men["in_cohort"]]

print("== verdict distribution =="); print(men["verdict"].value_counts())
print(f"unresolved share: {(men['verdict'] == 'UNCLEAR').mean():.0%} (ceiling 25%)")
check("vcep_parseable", coh["VariationID"].nunique(), "vcep_parseable")

print("\n== diagnostic: BA1 verdict stratified by classification ==")
ba1 = coh[coh["base"] == "BA1"]
print(pd.crosstab(ba1["bucket"], ba1["verdict"]))
bad = int(((ba1["verdict"].isin(APPLIED_LOOSE)) & (ba1["bucket"] == "VUS")).sum())
print(f"BA1 applied on a VUS (must be 0): {bad}")
if bad:
    DRIFT.append(f"BA1 applied on {bad} VUS")
men.to_csv(OUT_DIR / "code_mentions.csv", index=False)

## 6. Functional evidence by classification

The headline table. Percentages rest on 36–115 variants, so Wilson intervals accompany every point estimate.

In [ ]:
app = coh[coh["verdict"].isin(APPLIED_LOOSE)]
per_var = app.groupby(["VariationID","bucket","GeneSymbol"]).agg(
    codes=("code", lambda s: "; ".join(sorted(set(s))))).reset_index()
per_var["has_functional"] = per_var["codes"].apply(
    lambda s: any(c.strip().split("_")[0] in FUNCTIONAL for c in s.split(";")))

tab = per_var.groupby("bucket")["has_functional"].agg(with_functional="sum", curated="count")
print("== functional evidence applied, by classification ==")
for b in tab.index:
    k, n = int(tab.loc[b, "with_functional"]), int(tab.loc[b, "curated"])
    p, lo, hi = wilson(k, n)
    print(f"  {b:<5} {k:>3}/{n:<4} = {p:5.1%}   95% CI {lo:.1%}-{hi:.1%}")

for b, kk, nk in (("P/LP","plp_functional","plp_curated"),
                  ("B/LB","blb_functional","blb_curated"),
                  ("VUS","vus_functional","vus_curated")):
    if b in tab.index:
        check(f"{b} functional", int(tab.loc[b,"with_functional"]), kk)
        check(f"{b} curated",    int(tab.loc[b,"curated"]),         nk)

print("\n== PS3 versus BS3, unique variants ==")
func = coh[coh["base"].isin(FUNCTIONAL) & coh["verdict"].isin(APPLIED_LOOSE)]
print(func.groupby("base")["VariationID"].nunique())

print("\n== curation coverage against variant burden ==")
gene_tab = per_var.groupby("GeneSymbol").agg(
    curated=("VariationID","nunique"), with_functional=("has_functional","sum"))
gene_tab["missense_in_cohort"] = gene_tab.index.map(new["GeneSymbol"].value_counts())
gene_tab["pct_curated"] = (gene_tab["curated"]/gene_tab["missense_in_cohort"]*100).round(2)
print(gene_tab.sort_values("missense_in_cohort", ascending=False))

per_var.to_csv(OUT_DIR / "per_variant_applied.csv", index=False)
tab.to_csv(OUT_DIR / "headline_table.csv")
gene_tab.to_csv(OUT_DIR / "gene_curation_table.csv")

## 7. The blocked set

Variants the panel curated and left uncertain. An expert panel applied the specification to each and could not conclude, which makes this the object of the project.

Membership comes from curation status, not from extraction success. A worksheet that drops variants for being hard to parse is the wrong worksheet: those are the ones most in need of a human. Three tiers are tracked separately so a definition cannot drift silently.

In [ ]:
curated_vus = vg[vg["bucket"] == "VUS"][["VariationID","GeneSymbol"]].copy()
check("blocked_curated", len(curated_vus), "blocked_curated")

with_codes = set(coh.loc[coh["VariationID"].isin(curated_vus["VariationID"]), "VariationID"])
check("blocked_with_codes", len(with_codes), "blocked_with_codes")

applied_ids = set(per_var.loc[per_var["bucket"] == "VUS", "VariationID"])
check("blocked_with_applied", len(applied_ids), "blocked_with_applied")

blocked = curated_vus.copy()
blocked["extraction_status"] = blocked["VariationID"].map(
    lambda v: "applied_codes" if v in applied_ids
    else ("codes_unadjudicated" if v in with_codes else "no_codes_parsed"))
blocked = blocked.merge(per_var[["VariationID","codes","has_functional"]],
                        on="VariationID", how="left")
blocked["codes"] = blocked["codes"].fillna("")
blocked["has_functional"] = blocked["has_functional"].astype("boolean").fillna(False).astype(bool)

tmp = coh[coh["VariationID"].isin(blocked["VariationID"])].copy()
tmp["code_verdict"] = tmp["code"] + "[" + tmp["verdict"] + "]"
all_codes = (tmp.groupby("VariationID")["code_verdict"]
             .apply(lambda s: "; ".join(sorted(set(s))))
             .rename("codes_all_verdicts").reset_index())
blocked = blocked.merge(all_codes, on="VariationID", how="left")
blocked["codes_all_verdicts"] = blocked["codes_all_verdicts"].fillna("")

blocked = blocked.merge(new[["VariationID","Name","hgvs_p_short"]],
                        on="VariationID", how="left")
blocked["clinvar_url"] = blocked["VariationID"].map(
    lambda v: f"https://www.ncbi.nlm.nih.gov/clinvar/variation/{v}/")
for c in ["codes_applied_verified","codes_discussed_only","bare_list_means_applied",
          "binding_constraint","functional_assay_exists","assay_reference",
          "calibratable","notes"]:
    blocked[c] = ""

order = {"no_codes_parsed": 0, "codes_unadjudicated": 1, "applied_codes": 2}
blocked["_o"] = blocked["extraction_status"].map(order)
blocked = blocked.sort_values(["_o","GeneSymbol"]).drop(columns="_o").reset_index(drop=True)

print("== blocked set ==")
print(blocked["extraction_status"].value_counts())
print("\nby gene:"); print(blocked["GeneSymbol"].value_counts())
print(f"\nwith applied functional evidence   : {int(blocked['has_functional'].sum())}")
print(f"without applied functional evidence: {int((~blocked['has_functional']).sum())}")
blocked.to_csv(OUT_DIR / "blocked_vus_worksheet.csv", index=False)
N_BLOCKED = len(blocked)

## 8. MaveDB inventory

The accession hierarchy has three levels — `urn:mavedb:00000057` (experiment set), `-a` (experiment), `-a-1` (score set) — and only the last has a `/scores` endpoint. Counting all three inflated an earlier tally from 14 to 31, so the parsed count is cross-checked against the total the API reports for itself.

In [ ]:
SCORESET_RE = re.compile(r"urn:mavedb:\d{8}-[a-z0-9]+-\d+")

GENE_URNS, inv, shown = {}, [], False
for gene in GENES:
    status, data, _ = get_json(f"{MAVEDB}/genes/{gene}")
    if status == 200 and data is not None:
        if not shown:
            print("[diag ] first response keys:",
                  list(data.keys()) if isinstance(data, dict) else type(data))
            shown = True
        scoresets = sorted(set(SCORESET_RE.findall(json.dumps(data))))
        GENE_URNS[gene] = scoresets
        inv.append({"gene": gene, "status": 200, "n_scoresets": len(scoresets),
                    "scoresets": "; ".join(scoresets),
                    "total_reported": data.get("total"),
                    "scored_variants": data.get("totalScoredVariants")})
        print(f"[ok   ] {gene:<7} {len(scoresets)} score set(s)")
    else:
        GENE_URNS[gene] = []
        inv.append({"gene": gene, "status": status, "n_scoresets": 0, "scoresets": "",
                    "total_reported": None, "scored_variants": None})
        print(f"[warn ] {gene:<7} HTTP {status}")
    time.sleep(SLEEP)

inv = pd.DataFrame(inv)
inv["blocked_vus"] = inv["gene"].map(blocked["GeneSymbol"].value_counts()).fillna(0).astype(int)
inv.to_csv(OUT_DIR / "mavedb_inventory.csv", index=False)

print("\n== deposited score sets against blocked VUS ==")
print(inv[["gene","n_scoresets","total_reported","scored_variants","blocked_vus"]]
      .sort_values("blocked_vus", ascending=False).to_string(index=False))

mismatch = inv[inv["total_reported"].notna() & (inv["total_reported"] != inv["n_scoresets"])]
if len(mismatch):
    print("\n[WARN ] parsed count disagrees with the API's own total")
    print(mismatch[["gene","n_scoresets","total_reported"]].to_string(index=False))
else:
    print("\n[ok   ] parsed counts agree with the API's reported totals")


def ensure_scores(urn):
    p = SCORE_DIR / f"{urn.replace(':','_')}.csv"
    if p.exists():
        return pd.read_csv(p)
    try:
        r = requests.get(f"{MAVEDB}/score-sets/{urn}/scores", headers=UA, timeout=90)
        if r.status_code == 200:
            df = pd.read_csv(io.StringIO(r.text))
            df.to_csv(p, index=False)
            return df
        print(f"[warn ] {urn}: HTTP {r.status_code}")
    except Exception as exc:
        print(f"[fail ] {urn}: {type(exc).__name__}")
    return None

## 9. Coordinate resolution

A lookup by HGVS string returned zero coverage of the blocked set. That was wrong, and the reason generalises: MAVE depositions may number positions against the assay construct rather than the canonical protein.

Two independent routes are used. The first reconstructs the assay's own reference sequence from its `hgvs_pro` strings and slides it against the canonical sequence from UniProt. The second reads the offset MaveDB publishes in the score-set metadata. They should agree, and a mismatch means neither is trusted.

In [ ]:
def published_offset(urn):
    # MaveDB records the offset in targetGenes.externalIdentifiers. Reading it
    # is cheaper and more reliable than deriving it -- but deriving it
    # independently is what proves the reading is right.
    st, data, _ = get_json(f"{MAVEDB}/score-sets/{urn}")
    if st != 200 or not data:
        return None
    for tg in (data.get("targetGenes") or []):
        for ext in (tg.get("externalIdentifiers") or []):
            ident = ext.get("identifier", {})
            if ident.get("dbName") == "UniProt":
                return ext.get("offset")
    return None


def derived_offset(gene, acc, urns):
    if not urns:
        return None, 0.0
    df = ensure_scores(urns[0])
    if df is None or "hgvs_pro" not in df.columns:
        return None, 0.0
    ref = {}
    for s in df["hgvs_pro"].dropna().astype(str):
        m = re.match(rf"p\.({_AA})(\d+)", s)
        if m:
            ref[int(m.group(2))] = AA3_TO_1[m.group(1)]
    if not ref:
        return None, 0.0
    st, _, r = get_json(f"{UNIPROT}/{acc}.fasta")
    if r is None or r.status_code != 200:
        print(f"[warn ] UniProt {acc} unavailable")
        return None, 0.0
    canon = "".join(r.text.split("\n")[1:])
    scored = []
    for off in range(-200, 801):
        ok = tot = 0
        for p, aa in ref.items():
            c = p + off
            if 1 <= c <= len(canon):
                tot += 1
                ok += (canon[c-1] == aa)
        if tot >= max(20, 0.5 * len(ref)):
            scored.append((ok/tot, off, tot))
    scored.sort(reverse=True)
    print(f"\n{gene} ({acc}, {len(canon)} aa): assay covers {len(ref)} positions "
          f"{min(ref)}-{max(ref)}")
    for i, o, t in scored[:3]:
        print(f"   {i:6.1%}  offset {o:+5d}  n={t}")
    if scored and scored[0][0] > 0.95:
        return scored[0][1], scored[0][0]
    return None, (scored[0][0] if scored else 0.0)


OFFSETS = {}
for gene, acc in UNIPROT_BY_GENE.items():
    urns = GENE_URNS.get(gene, [])
    if not urns:
        continue
    derived, identity = derived_offset(gene, acc, urns)
    published = published_offset(urns[0])
    agree = (derived is not None and derived == published)
    note = ("agree" if agree else
            f"DISAGREE (derived {derived}, published {published})")
    print(f"   derived {derived}  |  published {published}  ->  {note}")
    if derived == 1:
        print("   an offset of +1 means the construct omits the initiator methionine")
    OFFSETS[gene] = derived if derived is not None else published
    if derived is not None and published is not None and not agree:
        DRIFT.append(f"{gene}: derived offset {derived} != published {published}")
    time.sleep(SLEEP)

print("\n== resolved offsets ==")
for g, o in OFFSETS.items():
    print(f"  {g:<6} {o if o is not None else 'UNRESOLVED'}")

## 10. Coverage and score distributions

Existence of a score set is not coverage of a variant, and a raw score means nothing without the distribution it sits in.

In [ ]:
recovered, dist_rows = [], []
for gene, off in OFFSETS.items():
    if off is None:
        continue
    sub = blocked[blocked["GeneSymbol"] == gene]
    if sub.empty:
        continue
    targets = {}
    for _, v in sub.iterrows():
        m = P_RE.search(str(v["Name"]))
        if not m:
            continue
        apos = int(m.group(2)) - off
        if apos < 1:
            # The canonical position falls before the start of the assay
            # construct. Reporting a negative coordinate would be arithmetically
            # correct and biologically meaningless: this is out of scope, not absent.
            print(f"    {v['hgvs_p_short']:<7} outside the construct "
                  f"(canonical {m.group(2)}, assay {apos})")
            continue
        targets[v["hgvs_p_short"]] = f"p.{m.group(1)}{apos}{m.group(3)}"
    for urn in GENE_URNS.get(gene, []):
        df = ensure_scores(urn)
        if df is None or "score" not in df.columns:
            continue
        df = df.dropna(subset=["score"])
        s = df["score"].astype(float)
        print(f"\n=== {gene} {urn}: {len(df):,} scored ===")
        print(f"    min {s.min():+.4f}  p5 {s.quantile(.05):+.4f}  median {s.median():+.4f}"
              f"  p95 {s.quantile(.95):+.4f}  max {s.max():+.4f}")
        for short, assay in targets.items():
            row = df[df["hgvs_pro"].astype(str) == assay]
            if not len(row):
                print(f"    {short:<7} {assay:<15} absent")
                continue
            v = float(row.iloc[0]["score"])
            pct = float((s < v).mean())
            print(f"    {short:<7} {assay:<15} score {v:+.4f}  percentile {pct:6.1%}")
            recovered.append({"gene": gene, "variant": short, "urn": urn,
                              "assay_hgvs": assay, "score": v})
            dist_rows.append({"gene": gene, "variant": short, "urn": urn,
                              "score": v, "percentile": pct, "n_scored": len(df)})

rec  = pd.DataFrame(recovered)
dist = pd.DataFrame(dist_rows)
rec.to_csv(OUT_DIR / "mavedb_recovered.csv", index=False)
dist.to_csv(OUT_DIR / "recovered_percentiles.csv", index=False)

n_cov = rec["variant"].nunique() if len(rec) else 0
print(f"\n{n_cov} of {N_BLOCKED} blocked VUS covered once coordinates are corrected")
print("(an HGVS-string lookup without the offset returns zero)")

## 11. Is calibration possible?

The precondition for an OddsPath estimate is not sample size but **separation**: do variants already classified pathogenic and benign occupy different parts of the score distribution? If they overlap, no amount of data yields an odds ratio.

ClinVar classifications are mapped into assay coordinates using the resolved offset, so like is compared with like.

In [ ]:
MIN_CONTROLS = 3
calib_rows = []

for gene, off in OFFSETS.items():
    if off is None or blocked[blocked["GeneSymbol"] == gene].empty:
        continue
    cv = new[new["GeneSymbol"] == gene].copy()
    cv["assay_hgvs"] = ("p." + cv["aa_ref"] + (cv["aa_pos"].astype(int) - off).astype(str)
                        + cv["aa_alt"])
    print(f"\n=== {gene}: {len(cv)} ClinVar missense ===")
    print(cv["bucket"].value_counts().to_string())

    for urn in GENE_URNS.get(gene, []):
        df = ensure_scores(urn)
        if df is None or "score" not in df.columns:
            continue
        df = df.dropna(subset=["score"]).copy()
        df["_key"] = df["hgvs_pro"].astype(str)
        jj = df.merge(cv[["assay_hgvs","bucket","Name"]],
                      left_on="_key", right_on="assay_hgvs", how="inner")
        if not len(jj):
            print(f"  {urn}: no classified variants present")
            continue
        jj["score"] = jj["score"].astype(float)
        print(f"\n  {urn}: {len(jj)} classified variants present")
        print(jj.groupby("bucket")["score"].agg(["count","median","min","max"]).round(4).to_string())

        plp = jj.loc[jj["bucket"] == "P/LP", "score"]
        blb = jj.loc[jj["bucket"] == "B/LB", "score"]
        row = {"gene": gene, "urn": urn, "n_plp": len(plp), "n_blb": len(blb)}
        if len(plp) >= MIN_CONTROLS and len(blb) >= MIN_CONTROLS:
            row["separation_testable"] = True
            try:
                from scipy.stats import mannwhitneyu
                u, p = mannwhitneyu(plp, blb, alternative="two-sided")
                row["mannwhitney_p"] = float(p)
                print(f"  => {len(plp)} P/LP vs {len(blb)} B/LB; Mann-Whitney p = {p:.4g}")
                print("     descriptive only: n is small and the controls are not")
                print("     independent of the assay literature")
            except Exception as exc:
                print(f"  [skip] {type(exc).__name__}")
        else:
            row["separation_testable"] = False
            print(f"  => only {len(plp)} P/LP and {len(blb)} B/LB present; "
                  f"separation cannot be assessed")
        calib_rows.append(row)

calib = pd.DataFrame(calib_rows)
calib.to_csv(OUT_DIR / "calibration_feasibility.csv", index=False)
print("\n== calibration feasibility ==")
print(calib.to_string(index=False) if len(calib) else "(nothing assessed)")
print("\nSeparation is a precondition, not a calibration. An OddsPath estimate")
print("requires controls selected independently of the assay; ClinVar labels")
print("derive in part from the same literature.")

## 12. What does the assay measure?

Coverage and separation do not settle whether evidence is usable. A readout in a heterologous system, scored for a single interaction, answers a mechanistic question; it does not license PS3 or BS3 for a germline variant.

In [ ]:
for gene in [g for g, o in OFFSETS.items() if o is not None]:
    for urn in GENE_URNS.get(gene, []):
        st, data, _ = get_json(f"{MAVEDB}/score-sets/{urn}")
        if st != 200 or not data:
            continue
        print("=" * 72)
        print(f"{gene}  {urn}\ntitle: {data.get('title')}")
        print("-" * 72)
        for k in ("shortDescription", "abstractText", "methodText"):
            if data.get(k):
                print(f"[{k}]\n{str(data[k])[:1200]}\n")
        for pub in (data.get("primaryPublicationIdentifiers") or
                    data.get("publicationIdentifiers") or []):
            print(f"[pub] {pub.get('dbName')} {pub.get('identifier')} — "
                  f"{str(pub.get('title',''))[:120]}")
        print()
        time.sleep(SLEEP)

## 13. Invariants and outputs

In [ ]:
print("=" * 64)
if DRIFT:
    print("DRIFT DETECTED -- investigate before citing any number:")
    for d in DRIFT:
        print("  -", d)
    print("\nDrift is not necessarily an error. ClinVar and MaveDB both change.")
    print("It means the frozen figures in this notebook no longer match the run.")
else:
    print("All invariants hold; results match the frozen figures.")
print("=" * 64)

print("\noutputs:")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {f.name:<36} {f.stat().st_size/1024:8.1f} KB")

## 14. Conclusions and limitations

Across 14 RAS/MAPK genes, functional evidence appears in roughly 55% of the panel's pathogenic classifications and 3% of its benign ones. A literature and database search across the same genes finds essentially no reported benign or neutral controls. Two measurements that share no methodology point at the same gap: assays in this space can support pathogenicity and have nothing to calibrate against on the benign side.

Deposited multiplexed assay data exist for two blocked VUS, in HRAS, invisible to string matching because the deposition numbers positions against the mature protein. That data is high quality for the question it was built for and cannot support PS3 or BS3 for germline classification without new calibration work. This is a fitness-for-purpose problem, not a scarcity problem.

### Limitations

1. **Detection, not absence.** The searches here establish what was found, not what exists. Manual review of the blocked records remains outstanding.
2. **Vocabulary-based detection undercounts**, and the undercount favours this project's thesis. Any claim resting on it should be read accordingly.
3. **The blocked set has three denominators** — curated, parseable, adjudicated. Percentages computed on one are not interchangeable with another.
4. **`APPLIED_BARE` is unresolved.** Whether a bare code list means "applied" changes the pathogenic count by roughly half. Both figures are reported.
5. **Small n.** Wilson intervals are given; point estimates alone are not reportable.
6. **VCEP attribution is a substring match**, and absence of a deposited classification is not absence of curation. Reclassifications made in clinical practice and never submitted are invisible here.
7. **Separation between existing ClinVar classifications is not calibration.** The labels derive in part from the same literature as the assays.
8. **Paralogue transfer is not an ACMG pathway.** PS1 and PM5 operate within a gene.

### Reproduction

Every input is public: two frozen monthly ClinVar releases, the ClinVar submission summary, the MaveDB API, and UniProt. No credentials, no private data. The invariant block reports any divergence from the frozen figures.